In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


objective: Build a model that predicts the house’s SalePrice as accurately as possible from the available features.

machine learning framing:   
- Type: Supervised learning  
- Task: Regression  
- Input: Numeric and categorical features (e.g., `OverallQual`, `GrLivArea`, etc.)  
- Output: Continuous variable (`SalePrice`)   

performance metric:  
- Evaluation metric: Root Mean Squared Error (RMSE)  
- Kaggle leaderboard uses RMSE in the original price space.  
- During training, we may apply a log-transform to `SalePrice` for numerical stability and back-transform predictions before submission.

goal:  
Minimize RMSE between predicted and actual sale prices.

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
import pandas as pd

# Load datasets
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
sample_submission = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')

# Display basic info
print("Train data shape:", train.shape)
print("Test data shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

In [ ]:
# Inspect the first few rows
train.head()

In [ ]:
# Overview of columns, data types, and missing values
train.info()

In [ ]:
# Check summary statistics for numerical features
train.describe().T.head(10)

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

I DO TASK 3 and 4 BELOW:

In [ ]:
#quick feature-type summary & counts
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load data (adjust path if needed)
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Basic shapes
print("train shape:", train.shape)
print("test shape:", test.shape)

# Feature type breakdown
dtype_counts = train.dtypes.value_counts()
print("\nData types count:\n", dtype_counts)

# Separate columns by pandas dtype
num_cols = train.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
print(f"\nNumeric columns: {len(num_cols)}")
print(f"Categorical (object) columns: {len(cat_cols)}")

# Show a few examples of each
print("\nNumeric sample:", num_cols[:10])
print("Categorical sample:", cat_cols[:10])

In [ ]:
# Missing values analysis
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.concat([missing, missing_pct], axis=1)
missing_df.columns = ['missing_count', 'missing_pct']
missing_df = missing_df[missing_df['missing_count'] > 0]

# Show features with missing values (descending)
display(missing_df.head(40))

# Features with many missing values (e.g., >30%)
high_missing = missing_df[missing_df['missing_pct'] > 30]
print("\nFeatures with >30% missing values:\n", high_missing)

In [ ]:
# Inspect unique counts for numeric columns that might be categorical-coded
num_unique = train[num_cols].nunique().sort_values()
print("Numeric columns with small number of unique values (may be ordinal/categorical):")
display(num_unique[num_unique < 25].head(60))

# Examples of bounded/ordinal columns
ordinal_candidates = ['OverallQual','OverallCond','BsmtFullBath','BsmtHalfBath','KitchenAbvGr']
print("\nOrdinal-like examples and their ranges:")
for c in ordinal_candidates:
    if c in train.columns:
        print(c, "->", sorted(train[c].unique())[:10], "(min,max)=", train[c].min(), train[c].max())

In [ ]:
# see: target distribution & skewness
# Target distribution
target = train['SalePrice']
print("SalePrice: mean =", target.mean(), "median =", target.median(), "skewness =", target.skew())

fig, axes = plt.subplots(1,2, figsize=(12,4))
sns.histplot(target, kde=True, ax=axes[0])
axes[0].set_title('SalePrice distribution (original)')

sns.histplot(np.log1p(target), kde=True, ax=axes[1])
axes[1].set_title('SalePrice distribution (log1p)')
plt.tight_layout()
plt.show()

Log-transform (log1p) makes distribution more Gaussian-like, helpful for linear models and stabilizes variance. Clearly show both plots in notebook.

In [ ]:
# How each numeric feature relates to log1p(SalePrice)
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Compute log1p SalePrice
train['LogSalePrice'] = np.log1p(train['SalePrice'])

# Get 38 correlated numeric features based on log1p(SalePrice)
top_corr = train.corr(numeric_only=True)['LogSalePrice'].abs().sort_values(ascending=False).head(38).index

# Compute grid size automatically
n_cols = 4
n_rows = math.ceil(len(top_corr) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

# Plot each feature against log1p(SalePrice)
for i, col in enumerate(top_corr):
    sns.scatterplot(x=train[col], y=train['LogSalePrice'], alpha=0.5, ax=axes[i])
    axes[i].set_title(f"{col} vs log1p(SalePrice)")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("log1p(SalePrice)")

# Hide any unused axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


I then created three new interaction features that combine house quality and size (out of my own curiosity, I think it should be make sense):
- `qualitylivingarea = OverallQual * GrLivArea`
- `qualityroom = OverallQual * TotRmsAbvGrd`

The goal is to capture how overall quality amplifies the effect of size and room count on price.  
Correlation analysis shows how these features have relationships with `SalePrice` below.

This is what Grok said when I proposed these new features:
- The plot of OverallQual vs. SalePrice shows a general upward trend, suggesting that higher quality ratings correlate with higher sale prices. Similarly, GrLivArea (above-grade living area) vs. SalePrice exhibits a positive correlation, with larger living areas generally associated with higher prices. Combining these into qualitylivingarea makes sense because it captures the interaction between quality and living space, which appears to have a stronger influence on price than either feature alone.
- The plot of TotRmsAbvGrd vs. SalePrice also shows some correlation, though with more variability and fewer extreme values compared to GrLivArea. Multiplying TotRmsAbvGrd by OverallQual to create qualityroom accounts for the quality of the rooms, which could refine the impact of room count on price. This is particularly relevant as the plot indicates that the number of rooms alone doesn’t consistently predict price without considering quality.

In [ ]:
# --- Create engineered features ---
train['qualitylivingarea'] = train['OverallQual'] * train['GrLivArea']
train['qualityroom'] = train['OverallQual'] * train['TotRmsAbvGrd']

In [ ]:
# Compute log1p SalePrice (if not already created)
train['LogSalePrice'] = np.log1p(train['SalePrice'])

# --- Correlation of new features with log1p(SalePrice) ---
corr_new_features = train[['qualitylivingarea', 'qualityroom', 'OverallQual', 'GrLivArea', 'TotRmsAbvGrd', 'LogSalePrice']].corr()

# Sort correlations by LogSalePrice
corr_new_features['LogSalePrice'].sort_values(ascending=False)

In [ ]:
# --- Scatter plots of engineered features vs log1p(SalePrice) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x='qualitylivingarea', y='LogSalePrice', data=train, alpha=0.5, ax=axes[0])
axes[0].set_title('qualitylivingarea vs log1p(SalePrice)')
axes[0].set_xlabel('qualitylivingarea')
axes[0].set_ylabel('log1p(SalePrice)')

sns.scatterplot(x='qualityroom', y='LogSalePrice', data=train, alpha=0.5, ax=axes[1])
axes[1].set_title('qualityroom vs log1p(SalePrice)')
axes[1].set_xlabel('qualityroom')
axes[1].set_ylabel('log1p(SalePrice)')

sns.scatterplot(x='OverallQual', y='LogSalePrice', data=train, alpha=0.5, ax=axes[2])
axes[2].set_title('OverallQual vs log1p(SalePrice)')
axes[2].set_xlabel('OverallQual')
axes[2].set_ylabel('log1p(SalePrice)')

plt.tight_layout()
plt.show()

I can see that the qualitylivingarea and qualityroom improve the correlation slightly above GrLivArea, and TotRmsAbvGrd. Even though it is lower correlation than OverallQual.

As a discrete rating (e.g., 1 to 10), OverallQual shows vertical clusters with significant variability in SalePrice within each rating level. This suggests that while there’s a strong positive correlation (0.817185), the discrete nature limits its ability to capture fine-grained price variations, leading to potential inaccuracies in prediction. The sparse data at higher ratings further reduces reliability.

qualitylivingarea, a continuous variable, exhibits a broad, somewhat linear spread of data points across a range of values. With a correlation of 0.800062, it shows a strong positive relationship with SalePrice. The continuous nature allows for more granular differentiation, potentially enabling more accurate predictions compared to OverallQual, as it can reflect subtle changes in living area impacting price.

qualityroom, likely a discrete or semi-continuous count of rooms, displays distinct vertical clusters with a clear upward trend. Its correlation of 0.784899 is slightly lower than the other two, but the clustering suggests a more consistent relationship with SalePrice within each room count. However, like OverallQual, its discrete nature may limit precision compared to a fully continuous variable.

Summary: OverallQual’s discrete nature and internal variability hinder its accuracy in predicting SalePrice, despite its highest correlation. qualitylivingarea, as a continuous variable, likely offers better predictive accuracy due to its finer granularity, while qualityroom lags slightly behind but still outperforms OverallQual in capturing consistent trends.

I then tried to Normalize them using MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# --- Normalize selected features (except SalePrice/LogSalePrice) ---
scaler = MinMaxScaler()

# Columns to normalize
cols_to_normalize = ['qualitylivingarea', 'qualityroom', 'OverallQual']

# Create normalized copies (with _norm suffix)
train[[f'{col}_norm' for col in cols_to_normalize]] = scaler.fit_transform(train[cols_to_normalize])

# --- Scatter plots of normalized engineered features vs log1p(SalePrice) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x='qualitylivingarea_norm', y='LogSalePrice', data=train, alpha=0.5, ax=axes[0])
axes[0].set_title('Normalized qualitylivingarea vs log1p(SalePrice)')
axes[0].set_xlabel('Normalized qualitylivingarea')
axes[0].set_ylabel('log1p(SalePrice)')

sns.scatterplot(x='qualityroom_norm', y='LogSalePrice', data=train, alpha=0.5, ax=axes[1])
axes[1].set_title('Normalized qualityroom vs log1p(SalePrice)')
axes[1].set_xlabel('Normalized qualityroom')
axes[1].set_ylabel('log1p(SalePrice)')

sns.scatterplot(x='OverallQual_norm', y='LogSalePrice', data=train, alpha=0.5, ax=axes[2])
axes[2].set_title('Normalized OverallQual vs log1p(SalePrice)')
axes[2].set_xlabel('Normalized OverallQual')
axes[2].set_ylabel('log1p(SalePrice)')

plt.tight_layout()
plt.show()


Perfect! I will use these features for training in the next step. The thesis? it capture more nuanced aspects of house quality than a single OverallQual rating. Reduce potential of overfitting by distributing predictive power across multiple sub-features. Improve generalization in a regression or tree-based model.

NOW: ADDING one categorical feature!

In [ ]:
# --- Plot Categorical Features vs log1p(SalePrice) ---
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Copy training data
df = train.copy()

# Ensure log1p(SalePrice) exists
df['LogSalePrice'] = np.log1p(df['SalePrice'])

# Select only object or category dtype columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Found {len(categorical_cols)} categorical features:")
print(categorical_cols)

# Pick a few top categorical features for visualization
top_categoricals = ['Neighborhood', 'HouseStyle', 'Exterior1st', 'KitchenQual', 'SaleCondition']

plt.figure(figsize=(18, 12))

for i, col in enumerate(top_categoricals, 1):
    if col in df.columns:
        plt.subplot(3, 2, i)
        # Sort categories by median LogSalePrice
        order = df.groupby(col)['LogSalePrice'].median().sort_values(ascending=False).index
        sns.boxplot(x=col, y='LogSalePrice', data=df, order=order)
        plt.xticks(rotation=45)
        plt.title(f"{col} vs log1p(SalePrice) (median order)")
        plt.xlabel(col)
        plt.ylabel('log1p(SalePrice)')

plt.tight_layout()
plt.show()


Features like KitchenQual and Neighborhood show the strongest influence on log1p(SalePrice), with clear median differences and relatively tight IQRs for higher categories, suggesting good predictive potential. Thus, I will use Neighborhood as categorical feature addition in this project.

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
# --- 0. Imports ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- 1. Define features and target ---
features = ['qualitylivingarea', 'qualityroom', 'Neighborhood']
target = 'LogSalePrice'

X = train[features]
y = train[target]

# --- 2. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Preprocessing ---
numeric_features = ['qualitylivingarea', 'qualityroom']
categorical_features = ['Neighborhood']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# --- 4. Models ---
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, solver='lsqr'),  # Fixed solver to avoid sparse_cg error
    'DecisionTree': DecisionTreeRegressor(random_state=42)
}

# --- 5. Train, predict, evaluate ---
results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', model)])
    
    # Fit model
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'R2': r2}
    
    print(f"{name}: RMSE={rmse:.4f}, R2={r2:.4f}")

# --- 6. Feature importance analysis ---
# Get feature names after one-hot encoding
onehot_feature_names = list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
all_features = numeric_features + onehot_feature_names

# Linear models: coefficients
for name in ['LinearRegression', 'Ridge']:
    coef = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', models[name])]).fit(X_train, y_train).named_steps['regressor'].coef_
    feature_importance = pd.Series(coef, index=all_features).sort_values(key=abs, ascending=False)
    print(f"\n{name} Feature Importance:")
    print(feature_importance)

# Decision tree: feature_importances_
tree_model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', models['DecisionTree'])]).fit(X_train, y_train).named_steps['regressor']
tree_importances = pd.Series(tree_model.feature_importances_, index=all_features).sort_values(ascending=False)
print("\nDecision Tree Feature Importance:")
print(tree_importances)


The linear models (Linear Regression and Ridge) perform best, with RMSE around 0.18 and R² about 0.826, indicating they capture most of the variation in log-transformed SalePrice. Ridge slightly improves over Linear Regression by slightly regularizing extreme coefficients. In both models, the categorical Neighborhood dominates feature importance, while qualitylivingarea has a moderate positive effect and qualityroom contributes modestly. The decision tree, by contrast, focuses almost entirely on qualitylivingarea, giving it over 80% of importance, while ignoring most neighborhood effects. This explains its higher RMSE and lower R²: it overfits numeric thresholds but fails to generalize across neighborhoods. Overall, linear models provide smoother, more reliable predictions by balancing neighborhood and numerical features, whereas the tree is too sensitive to limited features and splits.

EXPERIMENT: The high RMSE and over-reliance on qualitylivingarea in the decision tree indicate overfitting. With only three features, the tree can easily memorize numeric thresholds instead of generalizing. We can fix this by controlling the tree’s complexity with hyperparameters like max_depth, min_samples_split, and min_samples_leaf.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Fix the decision tree by limiting complexity
tree_model_fixed = DecisionTreeRegressor(
    random_state=42,
    max_depth=5,           # limit tree depth
    min_samples_split=10,  # minimum samples to split a node
    min_samples_leaf=5     # minimum samples in a leaf
)

# Build a pipeline with preprocessing
pipeline_tree = Pipeline(steps=[('preprocessor', preprocessor),
                                ('regressor', tree_model_fixed)])

# Fit model
pipeline_tree.fit(X_train, y_train)

# Predict and evaluate
y_pred_tree = pipeline_tree.predict(X_test)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))
r2_tree = r2_score(y_test, y_pred_tree)

print(f"Fixed Decision Tree: RMSE={rmse_tree:.4f}, R2={r2_tree:.4f}")

# Feature importance
onehot_features = list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
all_features = numeric_features + onehot_features
tree_importances = pd.Series(pipeline_tree.named_steps['regressor'].feature_importances_, index=all_features).sort_values(ascending=False)
print("\nFixed Decision Tree Feature Importance:")
print(tree_importances)


After constraining the decision tree with max_depth=5, min_samples_split=10, and min_samples_leaf=5, the model shows slightly improved generalization (RMSE = 0.2144, R² = 0.7537) compared to the unconstrained tree. However, qualitylivingarea overwhelmingly dominates feature importance (0.928), while Neighborhood and qualityroom contribute minimally, with most neighborhood features at or near zero. This indicates that even with complexity constraints, the tree primarily relies on numeric splits to predict SalePrice, largely ignoring categorical distinctions. Limiting the tree depth reduces overfitting but does not enable meaningful use of neighborhood information due to the small number of features. Overall, while the fixed tree smooths predictions slightly, linear or Ridge models remain superior for this feature set, as they effectively balance both numeric and categorical influences to achieve higher predictive accuracy.

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


Since the two best-performing models are Linear Regression and Ridge, we can fine-tune them (Ridge can benefit from hyperparameter tuning, while Linear Regression has almost no hyperparameters). For the decision tree, we can fine-tune the fixed tree to see if we can squeeze more performance. Here’s a complete approach using RandomizedSearchCV for Ridge and GridSearchCV for Decision Tree:

In [ ]:
# --- 0. Imports ---
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# --- 1. Ridge Hyperparameter Tuning with RandomizedSearchCV ---
ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', Ridge(solver='lsqr'))])

# Hyperparameter distribution
ridge_params = {
    'regressor__alpha': np.logspace(-3, 3, 100),  # from 0.001 to 1000
}

ridge_search = RandomizedSearchCV(
    ridge_pipeline,
    param_distributions=ridge_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)

# Predict and evaluate
y_pred_ridge = ridge_search.predict(X_test)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge = r2_score(y_test, y_pred_ridge)

print("Tuned Ridge:")
print(f"Best alpha: {ridge_search.best_params_['regressor__alpha']:.4f}")
print(f"RMSE={rmse_ridge:.4f}, R2={r2_ridge:.4f}")

# --- 2. Decision Tree Hyperparameter Tuning with GridSearchCV ---
tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('regressor', DecisionTreeRegressor(random_state=42))])

# Grid search parameters
tree_params = {
    'regressor__max_depth': [3, 5, 7, 10],
    'regressor__min_samples_split': [2, 5, 10, 20],
    'regressor__min_samples_leaf': [1, 2, 5, 10]
}

tree_search = GridSearchCV(
    tree_pipeline,
    param_grid=tree_params,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

tree_search.fit(X_train, y_train)

# Predict and evaluate
y_pred_tree = tree_search.predict(X_test)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))
r2_tree = r2_score(y_test, y_pred_tree)

print("\nTuned Decision Tree:")
print(f"Best params: {tree_search.best_params_}")
print(f"RMSE={rmse_tree:.4f}, R2={r2_tree:.4f}")


After comparing all models, the tuned Ridge regression emerges as the best choice for predicting log-transformed SalePrice. With a best alpha of 0.5337, it achieves RMSE = 0.1799 and R² = 0.8266, slightly outperforming untuned Linear Regression and significantly surpassing the decision tree in both accuracy and explained variance. Ridge effectively balances the contributions of numeric features (qualitylivingarea, qualityroom) and categorical information (Neighborhood), producing smooth, reliable predictions without overfitting. In contrast, the decision tree, even after tuning, over-relies on qualitylivingarea and fails to fully capture neighborhood effects, resulting in lower generalization. Therefore, for this dataset and feature set, Ridge regression is the most robust and predictive model.

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

Non-technical: We built a model to predict house prices using three key features: living area size, number of rooms, and neighborhood. The best model, Ridge regression, provides accurate and reliable price estimates while accounting for both the size of the house and its location. This allows your team to make data-driven decisions on property valuation and investment opportunities with confidence.

Technical: We engineered three predictive features, qualitylivingarea, qualityroom, and Neighborhood (one-hot encoded) and trained several models, including Linear Regression, Ridge, and Decision Tree. Hyperparameter tuning via RandomizedSearchCV optimized Ridge’s alpha to 0.5337, achieving RMSE = 0.1799 and R² = 0.8266 on the test set, outperforming tuned decision trees (RMSE = 0.1923, R² = 0.8017). Ridge effectively balances numeric and categorical features, producing smooth predictions with minimal overfitting, making it the most robust choice for property price estimation.

# 9. Submit your predictions to the competition

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# --- Load datasets ---
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# --- Engineer features ---
train['qualitylivingarea'] = train['GrLivArea'] * train['OverallQual']
train['qualityroom'] = train['TotRmsAbvGrd'] * train['OverallQual']

test['qualitylivingarea'] = test['GrLivArea'] * test['OverallQual']
test['qualityroom'] = test['TotRmsAbvGrd'] * test['OverallQual']

# --- Define features ---
final_features = ['qualitylivingarea', 'qualityroom', 'Neighborhood']

# --- Target ---
train['LogSalePrice'] = np.log1p(train['SalePrice'])
y_train = train['LogSalePrice']

# --- Preprocessing ---
numeric_features = ['qualitylivingarea', 'qualityroom']
categorical_features = ['Neighborhood']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# --- Tuned Ridge ---
final_ridge_model = Ridge(alpha=0.5337, solver='lsqr', random_state=42)

# --- Final pipeline ---
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', final_ridge_model)
])

# --- Fit pipeline ---
final_pipeline.fit(train[final_features], y_train)

# --- Predict on test ---
test_predictions_log = final_pipeline.predict(test[final_features])
test_predictions = np.expm1(test_predictions_log)  # reverse log1p

# --- Prepare submission ---
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True
    )
)

print("Submission file created successfully!")


In [ ]:
%%bash

cat submission.csv | head -n 5